In [55]:
#the sigmoid function

import numpy as np
z=np.array([-4,-2,0,2,4])
prob=1/(1+np.exp(-z))
print(prob.round(2))

[0.02 0.12 0.5  0.88 0.98]


In [56]:
import pandas as pd
rng=np.random.default_rng(42)
n=100

In [57]:
data=pd.DataFrame({
    "hours": rng.integers(1,11,n),
    "sleep": rng.integers(4,10,n),
    "attendence":rng.integers(60,101,n)})
noise=rng.normal(0,4,n)
marks=(10+4*data["hours"]+2*data["sleep"]+0.3*data["attendence"]+noise).round()
data["passed"]=(marks>=65).astype(int)
print(data.head())
print(data["passed"].value_counts())

   hours  sleep  attendence  passed
0      1      8          75       0
1      8      5          97       1
2      7      8          80       1
3      5      4          88       0
4      5      8          78       0
passed
1    64
0    36
Name: count, dtype: int64


In [58]:
from sklearn.model_selection import(train_test_split)

feats = ["hours", "sleep", "attendence"]

train , test = train_test_split(data, test_size = 0.25, random_state= 42, stratify=data["passed"])
y_train = train["passed"]
y_test = test["passed"]
print(len(train), len(test))
print(y_test.value_counts().to_dict())

75 25
{1: 16, 0: 9}


In [59]:
#Train and predict

from sklearn.linear_model import (LogisticRegression)

clf=LogisticRegression()
clf.fit(train[feats],y_train)
pred=clf.predict(test[feats])
print(pred[:10])
print(y_test.iloc[:10].values)

[1 1 1 0 1 1 0 1 1 1]
[1 1 1 0 1 1 0 1 1 1]


In [60]:
#the coefficients

coefs=pd.Series(clf.coef_[0],index=feats)
print(coefs.round(2))

hours         1.30
sleep         0.85
attendence    0.08
dtype: float64


In [61]:
student =pd.DataFrame({
    "hours":[6], "sleep":[8],
    "attendence":[90]})
print(clf.predict_proba(student).round(2))
print(clf.predict(student))

[[0.01 0.99]]
[1]


In [62]:
#confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm= confusion_matrix(y_test, pred)
acc= accuracy_score(y_test, pred)
print(cm)
print("Accuracy:",acc)

[[ 6  3]
 [ 0 16]]
Accuracy: 0.88


In [63]:
from sklearn.dummy import DummyClassifier

dummy=DummyClassifier(
 strategy="most_frequent")
dummy.fit(train[feats],y_train)
print(dummy.score(test[feats],y_test))

0.64


In [64]:
#accuracy can lie


import numpy as np
from sklearn.metrics import accuracy_score, recall_score

y_true = np.array([1] * 10 + [0] * 990)
y_pred = np.zeros(1000, dtype=int)

acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)

print("Accuracy:", acc)
print("Recall:", rec)


Accuracy: 0.99
Recall: 0.0


In [65]:
#IN CODE
from sklearn.metrics import(
    precision_score,f1_score)

prec=precision_score(y_test,pred)
rec=recall_score(y_test,pred)
f1=f1_score(y_test,pred)
print("precission:",round(prec,2))
print("Recall:",round(rec,2))
print("F1 :",round(f1,2))

precission: 0.84
Recall: 1.0
F1 : 0.91


In [70]:

from sklearn.metrics import precision_score, recall_score

# Calculate probability of class 1 (Passed)
proba = clf.predict_proba(test[feats])[:, 1]

# Test different thresholds
for th in [0.3, 0.5, 0.7, 0.9]:
    p = (proba >= th).astype(int)

    prc = precision_score(y_test, p, zero_division=0)
    rc = recall_score(y_test, p, zero_division=0)

    print(
        "Threshold:", th,
        "| Precision:", round(prc, 2),
        "| Recall:", round(rc, 2)
    )


Threshold: 0.3 | Precision: 0.84 | Recall: 1.0
Threshold: 0.5 | Precision: 0.84 | Recall: 1.0
Threshold: 0.7 | Precision: 0.94 | Recall: 0.94
Threshold: 0.9 | Precision: 1.0 | Recall: 0.88


In [71]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test,proba)
print("AUC:",round(auc,3))

AUC: 0.986


In [72]:
from sklearn.model_selection import(cross_val_score)

x,y = data[feats],data["passed"]
acc=cross_val_score(LogisticRegression(),x,y,cv=5)
print(acc.round(2))
print("Mean:",round(acc.mean(),3))

[0.85 1.   0.85 0.85 0.9 ]
Mean: 0.89


In [74]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import(StandardScaler)
pipe=make_pipeline(StandardScaler(),LogisticRegression())
f1s=cross_val_score(pipe,x,y,cv=5,scoring="f1")
print(f1s.round(3))
print("mean:",round(f1s.mean(),3))

[0.88  0.96  0.923 0.88  0.923]
mean: 0.913


In [76]:
#Compare classifiers

from sklearn.tree import(DecisionTreeClassifier)
from sklearn.neighbors import(KNeighborsClassifier)

models={
    "Logistic":pipe,
    "Tree":DecisionTreeClassifier(
        max_depth=3,random_state=42),
    "KNN":make_pipeline(
        StandardScaler(),
        KNeighborsClassifier()
    )
}
for name, m in models.items():
    s = cross_val_score(m, x, y, cv=5, scoring="f1")
    print(name, round(s.mean(), 3))

Logistic 0.913
Tree 0.831
KNN 0.903


In [77]:
#saving the model

import joblib
pipe.fit(x,y)
joblib.dump(pipe,"pass_model.pkl")
loaded=joblib.load("pass_model.pkl")
print(loaded.predict(student))

[1]
